# **Transfer Learning với ResNet50 cho Vietnamese Sign Language**

Notebook này sử dụng Transfer Learning với mô hình ResNet50 đã được huấn luyện trên ImageNet để phân loại 25 lớp ký hiệu ngôn ngữ ký hiệu Việt Nam.

## **I. Import thư viện và thiết lập tham số**

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

## **II. Cài đặt các thông số cơ bản (Hyperparameters)**

In [ ]:
# Đường dẫn tới dữ liệu đã được lưu từ notebook trước
TRAIN_PATH = '/content/drive/MyDrive/NTTU_Chuyen de AI_2/train_ds'
VAL_PATH = '/content/drive/MyDrive/NTTU_Chuyen de AI_2/val_ds'
TEST_PATH = '/content/drive/MyDrive/NTTU_Chuyen de AI_2/test_ds'

# Thông số mô hình
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 25
SEED = 42

print(f"Cấu hình:")
print(f"- Kích thước ảnh: {IMG_SIZE}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Số lớp: {NUM_CLASSES}")

## **III. Tải dữ liệu đã được chia sẵn**

In [ ]:
print("Đang tải dữ liệu từ Google Drive...")

# Tải các tập dữ liệu đã lưu
train_ds = tf.data.Dataset.load(TRAIN_PATH)
val_ds = tf.data.Dataset.load(VAL_PATH)
test_ds = tf.data.Dataset.load(TEST_PATH)

print(f"\nĐã tải dữ liệu thành công:")
print(f"- Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"- Val batches: {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"- Test batches: {tf.data.experimental.cardinality(test_ds).numpy()}")

## **IV. Tiền xử lý và tăng cường dữ liệu (Data Augmentation)**

In [ ]:
# Hàm tiền xử lý cho ResNet50
def preprocess_for_resnet(image, label):
    image = preprocess_input(image)
    return image, label

# Hàm tăng cường dữ liệu
def augment_data(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, label

# Áp dụng augmentation cho tập train
train_ds_augmented = train_ds.map(augment_data, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_augmented = train_ds_augmented.map(preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
train_ds_augmented = train_ds_augmented.prefetch(tf.data.AUTOTUNE)

# Chỉ tiền xử lý cho tập val và test (không augmentation)
val_ds_processed = val_ds.map(preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_processed = val_ds_processed.prefetch(tf.data.AUTOTUNE)

test_ds_processed = test_ds.map(preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
test_ds_processed = test_ds_processed.prefetch(tf.data.AUTOTUNE)

print("Đã áp dụng data augmentation và preprocessing.")

## **V. Xây dựng mô hình với Transfer Learning**

In [ ]:
print("Đang khởi tạo mô hình ResNet50...")

# Tải mô hình ResNet50 đã được huấn luyện trên ImageNet
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Đóng băng toàn bộ mạng gốc
base_model.trainable = False

# Gắn "Đầu" mới cho 25 lớp
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

print(f"\nTổng số layers: {len(model.layers)}")
print(f"Số layers trainable: {sum([1 for layer in model.layers if layer.trainable])}")
print(f"Số layers frozen: {sum([1 for layer in model.layers if not layer.trainable])}")

## **VI. Giai đoạn 1: Warmup - Huấn luyện lớp đầu**

In [ ]:
print("\n=== BẮT ĐẦU GIAI ĐOẠN 1: WARMUP ===")

# Hàm định nghĩa Learning Rate Warmup
def lr_warmup_scheduler(epoch, lr):
    warmup_epochs = 3
    target_lr = 1e-3
    if epoch < warmup_epochs:
        new_lr = target_lr * ((epoch + 1) / warmup_epochs)
        print(f"\n[Warmup] Epoch {epoch + 1}: Tăng LR lên {new_lr:.6f}")
        return new_lr
    else:
        print(f"\n[Warmup] Epoch {epoch + 1}: Giữ ổn định LR ở mức {target_lr:.6f}")
        return target_lr

warmup_callback = LearningRateScheduler(lr_warmup_scheduler)

# Biên dịch mô hình
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Huấn luyện 5 epoch chỉ với lớp đầu
history_stage1 = model.fit(
    train_ds_augmented,
    validation_data=val_ds_processed,
    epochs=5,
    callbacks=[warmup_callback]
)

print("\nHoàn tất Giai đoạn 1!")

## **VII. Giai đoạn 2: Fine-tuning - Tinh chỉnh toàn bộ mô hình**

In [ ]:
print("\n=== BẮT ĐẦU GIAI ĐOẠN 2: FINE-TUNING ===")

# Rã đông mô hình
base_model.trainable = True

print(f"Số layers trainable sau khi unfreeze: {sum([1 for layer in model.layers if layer.trainable])}")

# Biên dịch lại mô hình với SGD, tốc độ học rất nhỏ
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-5, momentum=0.9),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Các Callback bảo vệ mô hình
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# Huấn luyện sâu
history_stage2 = model.fit(
    train_ds_augmented,
    validation_data=val_ds_processed,
    epochs=30,
    callbacks=[early_stop, reduce_lr]
)

print("\nHoàn tất Giai đoạn 2!")

## **VIII. Đánh giá mô hình trên tập Test**

In [ ]:
print("\nĐang đánh giá mô hình trên tập Test...")

test_loss, test_accuracy = model.evaluate(test_ds_processed)

print(f"\n=== KẾT QUẢ TRÊN TẬP TEST ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## **IX. Vẽ biểu đồ quá trình huấn luyện**

In [ ]:
# Kết hợp lịch sử từ cả 2 giai đoạn
def combine_histories(hist1, hist2):
    combined = {}
    for key in hist1.history.keys():
        combined[key] = hist1.history[key] + hist2.history[key]
    return combined

full_history = combine_histories(history_stage1, history_stage2)

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(full_history['accuracy'], label='Train Accuracy')
axes[0].plot(full_history['val_accuracy'], label='Val Accuracy')
axes[0].axvline(x=5, color='red', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(full_history['loss'], label='Train Loss')
axes[1].plot(full_history['val_loss'], label='Val Loss')
axes[1].axvline(x=5, color='red', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## **X. Lưu mô hình**

In [ ]:
# Lưu mô hình
model_save_path = '/content/drive/MyDrive/NTTU_Chuyen de AI_2/resnet50_vsl_25_classes.h5'
model.save(model_save_path)

print(f"\nMô hình đã được lưu tại: {model_save_path}")
print("Hoàn tất!")

## **XI. Kiểm tra dự đoán trên một số mẫu**

In [ ]:
# Lấy tên các lớp
class_names = ['A', 'AA', 'B', 'C', 'D', 'DD', 'E', 'G', 'H', 'I', 'K', 'L', 'M', 
               'N', 'O', 'OOO', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Y']

# Lấy một batch từ tập test
for images, labels in test_ds.take(1):
    # Tiền xử lý ảnh
    processed_images = preprocess_input(images.numpy())
    
    # Dự đoán
    predictions = model.predict(processed_images)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Hiển thị 9 ảnh đầu tiên
    plt.figure(figsize=(12, 12))
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        true_label = class_names[labels[i].numpy()]
        pred_label = class_names[predicted_classes[i]]
        confidence = predictions[i][predicted_classes[i]] * 100
        
        color = 'green' if true_label == pred_label else 'red'
        plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)", color=color)
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()
    break